In [1]:
!pip install transformers


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
!pip install ipywidgets widgetsnbextension --upgrade

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 5.6 MB/s eta 0:00:01
   ---------------------------- ----------- 1.6/2.2 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 3.9 MB/s  0:00:00
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 6.9 MB/s  0:00:00

  Attempting uninstall: widgetsnbextension

    Found existing installation: widgetsnbextension 3.6.6

    Uninstalling widgetsnbextension-3.6.6:

      Successfully uninstalled widgetsnbextension-3.6.6

   ---------------------------------------- 0/3 [widgetsnbextension]
  Attempting uninstall: jupyterlab_widgets
   ---------------------------------------- 0/3 [widgetsnbextension]
    Found existing installation: jupyterlab-widgets 1.0.0
   ---------------------------------------- 0/3 [widgetsnbextension]
    Uninstalling jupyterlab-widgets-1.0.0:
  


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install accelerate


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import re

from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [5]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [6]:
train_data["dialogue"].sample(5)

5739     Charlie: simple question, where can i find som...
5314     Gemma: can't sleep?\r\nLara: I have to finish ...
6148     Amy: <file_photo>\nAmy: <file_photo>\nAmy: <fi...
14637    Henry: What does this leaning in thing mean?\r...
3991     James: Mike, I am fucking tired after this tri...
Name: dialogue, dtype: object

In [7]:
train_data.isna().sum()

id          0
dialogue    1
summary     0
dtype: int64

In [8]:
train_data=train_data.dropna()

In [9]:
train_data.shape

(14731, 3)

In [10]:
val_data.shape

(818, 3)

In [11]:
train_data=train_data.sample(n=5000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

# Preprocess Data

In [12]:
def clean_data(text):
    text=re.sub(r"[\r\n]"," ",text)
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r"<.*?>","",text)
    text=re.sub(r"[^\w\d]"," ",text)
    text=text.strip().lower()
    return text

In [13]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [14]:
train_data

,id,dialogue,summary
0,13811908,violet hi i came across this austin s articl...,violet sent claire austin s article
1,13731387,mel right it s time i bought some christmas ...,mel bought christmas presents while evie keeps...
2,13727910,carol can you pls pick up when i call you ca...,fiona can t talk to carol because she s very ...
3,13729823,adam do u have a map of paris tom yes why ...,tom has a map of paris
4,13681400,frank hi how s the family mike great sam ...,mike is happy because sam s moved out mike a...
...,...,...,...
4995,13727593,mom turn the volume down jack mom you can tu...,mom wants jack to turn the volume down
4996,13829310,claire i just got a job offer tom wow that ...,claire got a job offer for a paid social manag...
4997,13865225,sam have you heard about this guy sam geor...,george weah was a striker and in 2018 he was e...
4998,13681686,melinda how s your game teresa u mean dixit...,teresa hasn t played her dixit yet because she...


# Tokenize

In [15]:
Tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [16]:
def tokenize(data):
    inputs=Tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=Tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"]=targets["input_ids"]
    return inputs

In [17]:
training_data=train_data.apply(tokenize,axis=1).tolist()
validation_data=val_data.apply(tokenize,axis=1).tolist()

In [18]:
training_data[0]

{'input_ids': [25208, 7102, 3, 23, 764, 640, 48, 403, 17, 77, 3, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 3, 7997, 15, 7102, 2049, 68, 3, 23, 3, 162, 641, 608, 34, 3, 7997, 15, 68, 2049, 21, 1631, 81, 140, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [19]:
# input_ids ===> Tokenized Val of the inputs
# attention marks ===> 1 : Valid vals and 0:invalid vals
# labels ==>Invalid vals

In [20]:
type(training_data)

list

# WORKING WITH THE MODEL

In [21]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [22]:
import torch

if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"

print(device)

cuda


In [23]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [24]:
train_args=TrainingArguments(output_dir="./result",
                             num_train_epochs=5,
                             per_device_train_batch_size=8,
                             per_device_eval_batch_size=8,
                             eval_strategy="epoch",
                             save_strategy="epoch",
                             warmup_steps=500)

In [25]:
trainer=Trainer(model=model,
                args=train_args,
                train_dataset=training_data,
                eval_dataset=validation_data)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.028941,0.391788
2,0.431913,0.370818
3,0.404178,0.363364
4,0.383476,0.360744
5,0.378166,0.360007


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3125, training_loss=0.9777166516113281, metrics={'train_runtime': 4765.137, 'train_samples_per_second': 5.246, 'train_steps_per_second': 0.656, 'total_flos': 3383545036800000.0, 'train_loss': 0.9777166516113281, 'epoch': 5.0})

In [27]:
model.save_pretrained("./final_summary_model")
Tokenizer.save_pretrained("./final_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_summary_model\\tokenizer_config.json',
 './final_summary_model\\tokenizer.json')

In [28]:
model=T5ForConditionalGeneration.from_pretrained("./final_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./final_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Test the core logic of summarization

In [30]:
def summarizer(dialogue):
    # Clean Data
    dialogue=clean_data(dialogue)

    #Tokenize the dialogues
    inputs=tokenizer(dialogue,
                     padding="max_length",
                     max_length=512,
                     truncation=True,
                     return_tensors="pt")

    #Generate the summary
    targets=model.generate(input_ids=inputs["input_ids"],
                           attention_mask=inputs["attention_mask"],
                           max_length=150,
                           num_beams=4,   # Number of outputs ==> It means number of summaries
                           early_stopping=True)
    model.to(device)

    # Decode the Tokens
    summary=tokenizer.decode(targets[0],skip_special_tokens=True)
    return summary
                           


In [32]:
test_dialogue="""
A: Our exam schedule was announced today.
B: Really? When is the NLP exam?
A: It is next Monday morning.
B: Then we should start revising from tomorrow itself."""

summary=summarizer(test_dialogue)
print(summary)

a nlp exam schedule was announced today b will start revising from tomorrow
